# EDA - Global Mobile Reviews Dataset

Analisis Exploratorio de Datos de resenas de telefonos moviles.

In [19]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

from src.load_data import load_csv, summary_stats
from src.cleaning import adjust_dtypes, handle_missing, remove_duplicates
from src.analysis import descriptive_stats, correlation_analysis
from src.visualization import generate_all_visualizations

## Carga de Datos 
## FASE 0 : DATOS SIN PROCESAR

In [ ]:
df = load_csv('../data/dataset.csv')

In [ ]:
summary_stats(df)

## FASE 1 : Analisis Descriptivo

En esta fase calculamos estadísticas descriptivas por columna para entender la distribución y las medidas  (conteo, media, mediana, desviación estándar, min, max y percentiles).
La función `descriptive_stats(df)` devuelve un DataFrame `stats_df` con estos resultados y se usa para detectar sesgos y guiar la limpieza posterior.

In [ ]:
stats_df = descriptive_stats(df)

##  Limpieza y Transformacion

3.1 FASE 2 : Ajuste de tipos (`adjust_dtypes`) :
- Se convierte fechas a `datetime`, booleanos a `bool`, strings numéricos a `int/float` y columnas con pocas etiquetas a `category`.
- Ayuda a realizar cálculos, agrupaciones y visualizaciones ; reduce uso de memoria.

3.2 FASE 3 : Detección y tratamiento de datos ausentes (`handle_missing`):
- Se elimina columnas con muchos nulos (>50%). Imputa valores numéricos (por ejemplo mediana) y categóricos con la moda o `Unknown`.
- Por qué: evita errores en cálculos y modelos y mantiene calidad de datos.

3.3 Eliminacón de duplicados (`remove_duplicates`) :
- Elimina filas exactamente duplicadas y reporta cuántas se quitaron.
- Por qué: evita que duplicados inflen estadísticas y conteos.

Ejecuta la celda de abajo para aplicar estas tres operaciones y ver el resumen (`df.info()`).

In [ ]:
df = adjust_dtypes(df)
df = handle_missing(df, strategy='auto')
df = remove_duplicates(df)
df.info()

## FASE 4 :  Deteccion de Outliers

Identificamos valores atípicos en variables numéricas mediante el método IQR (rango intercuartílico).
Se reporta el número y porcentaje de outliers por columna para decidir acciones (imputación, recorte o conservación).
La revisión manual de ejemplos y la comprensión del contexto de la variable son clave antes de eliminar valores.

In [24]:
from src.cleaning import detect_outliers_iqr
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
outliers = detect_outliers_iqr(df, numeric_cols)
for col, info in outliers.items():
    if info['count'] > 0:
        print(f"{col}: {info['count']} outliers ({info['pct']}%)")

edad: 190 outliers (0.38%)
tasa_de_cambio_a_usd: 6124 outliers (12.25%)
votos_utiles: 913 outliers (1.83%)


## FASE 5 : Correlacion de Variables

## 5.1. Selección de variables numéricas
```python
df.select_dtypes(include=np.number)
```
✔ Toma solo columnas numéricas (precio, calificación, etc.)

---

## 5.2. Cálculo de correlación
```python
df.corr()
```
✔ Mide relación entre variables:
- 1 = relación fuerte
- 0 = sin relación
- -1 = relación inversa

---

## 5.3. Detección de relaciones fuertes
```python
abs(r) > 0.7
```
✔ Identifica variables muy relacionadas

---

## 5.4. Interpretación automática
- > 0.8 → muy fuerte  
- > 0.7 → fuerte  

---

## 5.5. Caso clave: precio vs satisfacción
✔ Si r ≈ 0:
 El precio no influye en la calificación

---

##  Idea principal
 El código detecta relaciones importantes y muestra si variables como precio influyen o no en la satisfacción del usuario.

In [ ]:
corr = correlation_analysis(df)

# Interpretación de la Matriz de Correlación

- La `calificacion` general tiene una correlación positiva fuerte con:
  - `calificacion_duracion_bateria` (0.762)
  - `calificacion_camara` (0.761)
  - `calificacion_pantalla` (0.757)
  - `calificacion_rendimiento` (0.756)
  - `calificacion_disenho` (0.755)

### Interpretación
La satisfacción de los usuarios depende principalmente de las características y experiencia de uso del smartphone.

---

- La variable `precio_usd` presenta una correlación muy baja con `calificacion` (0.001).

### Interpretación
El precio no muestra una relación significativa con la satisfacción general de los usuarios.  
Esto sugiere que los usuarios valoran más el rendimiento y las características del dispositivo que su costo.

---

- `votos_utiles` presenta una correlación moderada con `calificacion` (0.460).

### Interpretación
Las reseñas con mejores calificaciones tienden a recibir más votos útiles.

---

In [ ]:
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
plt.figure(figsize=(12, 10))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, linewidths=0.5)
plt.title('Matriz de Correlacion', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Visualizaciones

In [ ]:
generate_all_visualizations(df, corr)

## 7. Visualizaciones Adicionales en Notebook

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

brand_counts = df['brand'].value_counts()
axes[0,0].barh(brand_counts.index, brand_counts.values, color='steelblue')
axes[0,0].set_title('Resenas por Marca')

country_counts = df['country'].value_counts().head(10)
axes[0,1].bar(country_counts.index, country_counts.values, color='steelblue')
axes[0,1].set_title('Top 10 Paises')
axes[0,1].tick_params(axis='x', rotation=45)

ages = df['age']
axes[0,2].hist(ages, bins=30, edgecolor='white', color='steelblue')
axes[0,2].set_title('Distribucion de Edad')

ratings = df['rating']
axes[1,0].hist(ratings, bins=5, edgecolor='white', color='steelblue', align='left')
axes[1,0].set_title('Distribucion de Rating')
axes[1,0].set_xticks([1,2,3,4,5])

source_counts = df['source'].value_counts()
axes[1,1].pie(source_counts.values, labels=source_counts.index, autopct='%1.1f%%', startangle=90)
axes[1,1].set_title('Fuente de Resenas')

sns.boxplot(data=df, x='sentiment', y='price_usd', ax=axes[1,2], palette='Set2')
axes[1,2].set_title('Precio por Sentimiento')

plt.tight_layout()
plt.show()

In [ ]:
df_clean = df.copy()
df_clean.to_csv('../data/clean_data.csv', index=False)
print(f"Dataset limpio exportado: {df_clean.shape}")